# Week 2, day 5 (morning) — Worksheet 11 SOLUTIONS: scope and side effects   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q5 and Q8 are the two that matter. Both produce a correct-looking result and
change something the caller did not expect.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 11 — Scope and side effects. Run this once.
TAX_RATE = 0.20

orders_2019 = [2.50, 4.50, 2.00, 2.75, 2.50, 5.00, 8.00, 2.25]
orders_2020 = [3.50, 7.50, 1.10, 1.75, 3.50, 6.30, 4.60, 5.25]
orders_2021 = [4.00, 4.00, 9.90, 2.10, 3.30]

master_list = ["ana", "bo", "cai"]

print("TAX_RATE:", TAX_RATE)
print("master_list:", master_list)

PART A — Where names are looked up

### Question 1

Reading a global. -> `120.0` from both versions.

Same answer, and only one of them can be trusted.

`with_tax` found `TAX_RATE` by looking **outward**: not a local, not a
parameter, so Python checked the enclosing module and found it. That is the
G in LEGB, and it is how `PI` worked in worksheet 09 Q1.

`with_tax_explicit` was told. Its answer depends only on what it was
passed, which means you can read it, test it and move it to another file
without thinking. Q3 and Q8 are what happens to the other one.

(`TAX_RATE` prints as `0.2`, not `0.20` — the trailing zero was never
stored, as in worksheet 01 Q2.)

In [ ]:
def with_tax(amount):
    return amount * (1 + TAX_RATE)      # TAX_RATE is not a parameter

def with_tax_explicit(amount, rate):
    return amount * (1 + rate)

print(with_tax(100))
print(with_tax_explicit(100, TAX_RATE))

### Question 2

Assigning inside a function. -> `before: 0.2`, `inside: 0.25`, **`after: 0.2`**.

The function did not change the global. It made a **new local variable that
happens to share the name**, used it, and threw it away when the call
ended.

The rule is simple and catches everyone: **reading a name looks outward,
assigning to a name creates a local.** One `=` anywhere in the body is
enough — Python decides at compile time that the name is local for the
whole function, before a single line runs.

Which is why `TAX_RATE = TAX_RATE + 0.05` inside a function raises
`UnboundLocalError` rather than reading the global: the name is already
local by the time the right-hand side is evaluated, and it has no value
yet.

In [ ]:
print("before:", TAX_RATE)

def bump_rate():
    TAX_RATE = 0.25                     # assignment -- makes a NEW local name
    print("inside:", TAX_RATE)

bump_rate()
print("after: ", TAX_RATE)

### Question 3

`global`. -> `before: 0.2`, `after: 0.25`; then **`with_tax(100) is now 125.0`** while `with_tax_explicit(100, 0.20) is still 120.0`.

`global TAX_RATE` says "do not make a local, assign to the module-level
one". It works, and look at the damage.

`with_tax` was not edited. It was not called by `bump_rate_global`. Its
argument did not change. And its answer went from 120.0 to 125.0, because a
third function reached into the module and moved something underneath it.

`with_tax_explicit` could not be affected, because everything it uses
arrives through its parameters.

That is the case against `global` in one output. It is not that it fails —
it is that a function's behaviour stops being a property of the function.
Pass the value in, or return the new value and let the caller reassign it.

(The last line puts `TAX_RATE` back to 0.20. Without it, every later
question in the notebook would quietly use 0.25 — which is the same class
of problem all over again.)

In [ ]:
print("before:", TAX_RATE)

def bump_rate_global():
    global TAX_RATE
    TAX_RATE = 0.25

bump_rate_global()
print("after: ", TAX_RATE)

print("with_tax(100) is now", with_tax(100))
print("with_tax_explicit(100, 0.20) is still", with_tax_explicit(100, 0.20))

TAX_RATE = 0.20                         # put it back, or later questions drift

PART B — What a function can change behind your back

### Question 4

Reassigning a parameter. -> `inside: CHANGED`, `after: ana`.

A parameter is a local variable like any other. `name = "CHANGED"` pointed
the local name at a new string; the caller's `who` still points at the old
one.

So the caller is safe — and it is very easy to conclude from this that
arguments are copied and functions cannot touch what you pass them. Q5 is
why that conclusion is wrong.

In [ ]:
def rename(name):
    name = "CHANGED"
    print("inside:", name)

who = "ana"
rename(who)
print("after: ", who)

### Question 5

Mutating a parameter. -> `before: ['ana', 'bo', 'cai']`, `inside: [… 'dee']`, **`after: ['ana', 'bo', 'cai', 'dee']`**.

Same shape as Q4, opposite outcome. The caller's list changed.

The difference is **assignment versus mutation**, not string versus list.
`name = "CHANGED"` rebinds the local name and leaves the original object
alone. `names.append("dee")` does not rebind anything — it reaches into the
object both names point at and modifies it. There is only one list, and
the function was handed a second label for it.

Strings, numbers and tuples cannot be mutated at all, so they can never be
changed this way. Lists, dicts and sets can.

This is worksheet 02's aliasing, arriving through a function signature.
And it is invisible at the call site: `add_name(master_list)` looks exactly
like `rename(who)`, returns `None` in both cases, and only one of them
edited your data.

In [ ]:
print("before:", master_list)

def add_name(names):
    names.append("dee")
    print("inside:", names)

add_name(master_list)
print("after: ", master_list)

### Question 6

Copy versus in-place. -> `copy -- original: ['x', 'y'] returned: ['x', 'y', 'z']`; `inplace -- original: ['x', 'y', 'z'] returned: None`.

Two functions, two honest contracts:

- **`added_copy` returns a new list** and leaves the input untouched. Safe
  to call twice, safe to call on data you do not own, safe in a
  comprehension. `names + [new]` builds a new list rather than modifying
  one.
- **`added_inplace` modifies the input** and returns `None` — exactly like
  `list.append`, `list.sort` and `dict.update`, all of which return `None`
  for precisely this reason.

The returning-`None` part is a convention worth respecting: **a function
that mutates should not also return the thing it mutated**, because then
`b = added_inplace(a, "z")` gives you two names for one list and nobody can
tell which style the function is.

Pick one per function and say which in the docstring. The dangerous case is
the function that does both.

In [ ]:
def added_copy(names, new):
    return names + [new]                # + builds a new list

def added_inplace(names, new):
    names.append(new)                   # mutates, returns None

a = ["x", "y"]
result_a = added_copy(a, "z")
print("copy   -- original:", a, "returned:", result_a)

b = ["x", "y"]
result_b = added_inplace(b, "z")
print("inplace -- original:", b, "returned:", result_b)

PART C — The case the deck is actually making

### Question 7

The deck's own refactor. -> `Average order size in 2019 is 3.6875!`, `2020 is 4.1875!`, `2021 is 4.66!`.

Slides 44 and 48 write this out twice, with `sum_2019`/`cnt_2019` and then
`sum_2020`/`cnt_2020`. Three years the deck's way is 21 lines and three
chances to mistype a variable name; the classic version of that bug is the
second block still saying `sum_2019` somewhere, which produces a number
rather than an error.

Here the calculation exists once. A third year cost one line, and fixing a
bug in the averaging would cost one edit rather than three.

Note it both prints **and** returns. That is a compromise: the print is
what slides 44–48 asked for, and the `return` is what makes the function
usable by anything other than a human reading the output. If you only need
one, keep the return — worksheet 09 Q2.

In [ ]:
def report_average(year, orders):
    total = 0.00
    count = 0
    for order in orders:
        total = total + order
        count = count + 1
    average = total / count
    print(f"Average order size in {year} is {average}!")
    return average

report_average(2019, orders_2019)
report_average(2020, orders_2020)
report_average(2021, orders_2021)

# The deck's version is 7 lines per year: 21 for three years, 70 for ten.
# This is 8 lines once, plus 1 per year: 11 for three, 18 for ten -- and
# there is only one place a bug can live.

### Question 8

A global that moves. -> `90.0`, then `50.0`.

The function was not edited between the two calls. The argument was
identical. The answer changed by 40.

Two things are worth separating here. First, `DISCOUNT` did not have to
exist when the function was **defined** — only when it was **called**.
Python looks names up at call time, so a function can reference a global
that has not been created yet, and will raise `NameError` only if it is
still missing when someone calls it.

Second, and worse: `discounted(100)` is not a question with one answer. It
depends on the state of the module at the moment you ask. You cannot test
it without setting up that state, you cannot read the call site and know
what it will do, and a bug shows up as "the number was wrong on Tuesday".

`def discounted(price, discount):` costs one word at each call site and
removes the entire problem. That is what "pure" means, and it is why Q1's
second version was worth writing.

In [ ]:
def discounted(price):
    return price * (1 - DISCOUNT)

DISCOUNT = 0.10                 # defined AFTER the function, and it still works
print(discounted(100))

DISCOUNT = 0.50
print(discounted(100))

### Question 9

LEGB. -> `inner sees:  local`, `outer sees:  enclosing`, `module sees: global`.

Three variables, one name, three values alive at the same time. Python
resolves a name by searching four scopes in order:

**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.

`inner` stopped at Local. `outer` had no local `value` of its own at the
point it printed — its own assignment *is* the enclosing scope from
`inner`'s point of view — and the module-level one was never touched by
either.

The **B** at the end is the one that bites in real code: your own variable
called `list`, `sum`, `id` or `type` shadows the builtin, and the error
turns up later and somewhere else as `TypeError: 'int' object is not
callable`. Name a variable `sum` and you have disabled `sum()` for the rest
of that scope.

In [ ]:
value = "global"

def outer():
    value = "enclosing"

    def inner():
        value = "local"
        print("inner sees: ", value)

    inner()
    print("outer sees: ", value)

outer()
print("module sees:", value)

### Question 10

The namespace itself. -> `locals: ['a', 'b', 'c']`, `3`, `'c' in globals(): False`.

`locals()` is a real dictionary of everything alive in the call: the two
parameters and the one variable the body created. Parameters are locals —
there is no separate category.

`c` is not in `globals()` and never was. When the call ends the whole
namespace goes with it, which is exactly what Q11 runs into.

This is a debugging tool, not something to write into a program. But
printing `locals()` inside a function that is misbehaving is a genuinely
fast way to see what it actually received, as opposed to what you meant to
pass it.

In [ ]:
def inspect_me(a, b):
    c = a + b
    print("locals:", sorted(locals()))
    return c

print(inspect_me(1, 2))
print("'c' in globals():", "c" in globals())

### Question 11

Reaching for a local after the call. -> `inside: 42`, then `NameError: name 'answer' is not defined`.

The function ran correctly and printed the right number. The variable still
does not exist out here — it was created when the call started and
destroyed when it returned.

The fix is `return answer` and `answer = compute()`. That is worksheet 09
Q2 from the other side: a function that only prints has no way to hand
anything back, and a notebook makes it look like it did because the output
is right there on the screen.

`global answer` would also make this line work, and it is the wrong answer.
It is Q3's problem with extra steps — the value now lives in the module,
any function can move it, and you cannot call `compute()` twice for two
different results.

**Parameters in, return value out.** Everything else on this sheet is a
reason for that one sentence.

In [ ]:
def compute():
    answer = 42
    print("inside:", answer)

compute()

# This is SUPPOSED to raise: NameError. `answer` was created when the call
# started and destroyed when it finished. It never existed out here.
#
# The function should have RETURNED it -- `return answer` -- and the caller
# should have caught it: `answer = compute()`. That is worksheet 09 Q2 from
# the other side: a function that only prints has no way to hand anything
# back, and `global answer` is not the fix, it is the same bug with a
# licence.
print(answer)